# Penyimpanan Model
Menggabungkan pipeline (notebook 03) dan konfigurasi risk tier (notebook 04) menjadi satu artefak `models/v1/vendai_model.pkl` untuk deploy, lalu memverifikasi artefak tersebut bekerja mandiri pada data mentah.

# Import Library

In [1]:
import os
import sys
import json
import platform
from datetime import date
import pandas as pd
import joblib
import sklearn
from sklearn.metrics import roc_auc_score, f1_score

sys.path.append('..')
from app.risk_scoring import assign_risk_tier

# Load Artefak

In [2]:
pipeline = joblib.load('../models/v1/model_pipeline.pkl')

with open('../models/v1/risk_tier_config.json') as f:
    config = json.load(f)

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')['Risk_Level']

# Metrik Final pada Data Test

In [3]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'roc_auc': round(roc_auc_score(y_test, y_proba), 4),
    'f1': round(f1_score(y_test, pipeline.predict(X_test)), 4),
}
metrics

c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\HP\anaconda3\envs\capstone_pr

{'roc_auc': 0.9341, 'f1': 0.8925}

# Simpan Artefak Final

In [4]:
bundle = {
    'pipeline': pipeline,
    'risk_tier_config': config,
    'metadata': {
        'features': list(pipeline.feature_names_in_),
        'model': type(pipeline.named_steps['model']).__name__,
        'metrics_test': metrics,
        'sklearn_version': sklearn.__version__,
        'python_version': platform.python_version(),
        'created': date.today().isoformat(),
    },
}

joblib.dump(bundle, '../models/v1/vendai_model.pkl')
bundle['metadata']

{'features': ['Financial_Stability_Score',
  'On_Time_Delivery_Rate',
  'Defect_Rate',
  'Lead_Time_Days',
  'Alternative_Suppliers_Available',
  'Contract_Length_Months',
  'Environmental_Compliance',
  'Previous_Disruptions'],
 'model': 'HistGradientBoostingClassifier',
 'metrics_test': {'roc_auc': 0.9341, 'f1': 0.8925},
 'sklearn_version': '1.7.0',
 'python_version': '3.11.16',
 'created': '2026-11-08'}

# Verifikasi Artefak
Muat ulang dari file seperti saat deploy, lalu pastikan hasilnya identik dengan pipeline asli.

In [5]:
loaded = joblib.load('../models/v1/vendai_model.pkl')

reloaded_proba = loaded['pipeline'].predict_proba(X_test[loaded['metadata']['features']])[:, 1]
assert (reloaded_proba == y_proba).all(), 'Hasil model setelah dimuat ulang berbeda'

print('Artefak valid. ROC-AUC:', roc_auc_score(y_test, reloaded_proba))

Artefak valid. ROC-AUC: 0.9341125371335367


# Simulasi Data Vendor Baru
Data mentah seperti yang akan dikirim sistem rekomendasi: ada nilai kosong (`None`) dan kolom tambahan (`Nama_Vendor`) yang tidak dipakai model.

In [6]:
vendor_baru = pd.DataFrame([
    {'Supplier_ID': 'NEW_001', 'Nama_Vendor': 'Vendor A', 'Financial_Stability_Score': 90.0, 'On_Time_Delivery_Rate': 95.0,
     'Defect_Rate': 0.8, 'Lead_Time_Days': 14, 'Alternative_Suppliers_Available': 5, 'Contract_Length_Months': 12,
     'Environmental_Compliance': 90.0, 'Previous_Disruptions': 0},
    {'Supplier_ID': 'NEW_002', 'Nama_Vendor': 'Vendor B', 'Financial_Stability_Score': 90.0, 'On_Time_Delivery_Rate': 70.0,
     'Defect_Rate': 5.0, 'Lead_Time_Days': 60, 'Alternative_Suppliers_Available': 5, 'Contract_Length_Months': 12,
     'Environmental_Compliance': 90.0, 'Previous_Disruptions': 3},
    {'Supplier_ID': 'NEW_003', 'Nama_Vendor': 'Vendor C', 'Financial_Stability_Score': 35.0, 'On_Time_Delivery_Rate': 65.0,
     'Defect_Rate': 6.0, 'Lead_Time_Days': None, 'Alternative_Suppliers_Available': 0, 'Contract_Length_Months': 36,
     'Environmental_Compliance': None, 'Previous_Disruptions': 4},
])

hasil_baru = assign_risk_tier(loaded['pipeline'], vendor_baru, loaded['risk_tier_config'])
hasil_baru.insert(0, 'Supplier_ID', vendor_baru['Supplier_ID'])

assert list(hasil_baru['Risk_Tier']) == ['Rendah', 'Sedang', 'Tinggi'], 'Tier simulasi tidak sesuai kriteria'
hasil_baru

,Supplier_ID,Risk_Score,Operasional_Buruk,Teknis_Buruk,Kriteria_Teknis_Gagal,Risk_Tier
0,NEW_001,0.012360,False,False,,Rendah
1,NEW_002,0.038407,False,True,"On_Time_Delivery_Rate, Defect_Rate, Lead_Time_...",Sedang
2,NEW_003,0.999925,True,True,"On_Time_Delivery_Rate, Defect_Rate, Previous_D...",Tinggi


In [7]:
os.listdir('../models/v1')

['model_pipeline.pkl', 'risk_tier_config.json', 'vendai_model.pkl']